# Three spacecraft approaching Base, with JWST as a reference

Run on Windows with your existing **LIFEpy / STK 13** environment. Keep `2025-11-06_horizons_JWST_hourly_from-launch.e` in the notebook working directory.

- **JWST:** the supplied ephemeris remains loaded and visible throughout the scenario, as a magenta star and trajectory.
- **Base:** starts at JWST's position and velocity, then coasts independently.
- **X_Minus and X_Plus:** start at −150 m and +150 m relative to Base. Each receives one small inward burn, with a small opposite Z component to pass beside Base.

The nominal approach time is **6 hours**, with an approximate **5 m** offset from Base. The scenario lasts **9 hours**, including the departure after the pass. There is no braking burn, docking, or ongoing station keeping. JWST's imported trajectory includes whatever real manoeuvres are represented in its ephemeris; no burns are added to it here.

Run cells **1–9** to build the scenario and apply the burns. Cell **10** starts or replays the approach; cell **11** pauses near the expected close pass and zooms in. Cell **12** saves the current scenario. Running all cells leaves the view at the nominal close pass; rerun cell 10 to watch from the beginning.

The notebook's structure and Python syntax have been checked. Actual STK propagation and graphics must run on your Windows installation.


## 1. Imports

Select the same Python environment that worked for the previous project. Import the standard Python modules and STK 13 API.

In [1]:
import os
from datetime import datetime, timedelta
from pathlib import Path

if os.name != 'nt':
    raise SystemExit('Run this notebook on Windows with STK Desktop installed.')

from agi.stk13.stkdesktop import STKDesktop
from agi.stk13.stkobjects import AgESTKObjectType, AgEVePropagatorType, AgEVeProfile
from agi.stk13.stkobjects.astrogator import AgEVASegmentType, AgEVAElementType, AgEVAManeuverType, AgEVAAttitudeControl


## 2. User configuration

Change `folder` if your notebook kernel starts in a different directory. `separation_m` is the initial distance between the outer spacecraft. `miss_distance_m` is an approximate aiming offset, not an enforced minimum separation. The simple velocity estimate is distance divided by approach time; orbital dynamics slightly change the resulting encounter.


In [2]:
folder = Path.cwd()  # Notebook working directory; change this path if needed
ephemeris_file = folder / '2025-11-06_horizons_JWST_hourly_from-launch.e'
start_time = '6 Oct 2022 07:00:00.000'  # Same Base departure epoch as the previous project
separation_m = 300.0  # Initial distance BETWEEN the outer spacecraft
approach_hours = 6.0
miss_distance_m = 5.0  # Approximate Z offset from Base at the close pass
propagator_name = 'Earth Default High Fidelity v13'

if not ephemeris_file.is_file():
    raise FileNotFoundError(f'Place the supplied JWST .e file in this folder: {ephemeris_file}')
if approach_hours <= 0 or separation_m <= 0 or miss_distance_m <= 0:
    raise ValueError('approach_hours, separation_m and miss_distance_m must be positive.')

approach_seconds = approach_hours * 3600
coast_seconds = approach_seconds * 1.5
duration_days = coast_seconds / 86400
start_date = datetime.strptime(start_time, '%d %b %Y %H:%M:%S.%f')
stop_time = (start_date + timedelta(seconds=coast_seconds)).strftime('%d %b %Y %H:%M:%S.%f')[:-3]
close_time = (start_date + timedelta(seconds=approach_seconds)).strftime('%d %b %Y %H:%M:%S.%f')[:-3]


## 3. Start a new STK application and create the scenario

Open a new STK window and set the scenario dates and units. Run this cell once per new scenario; rerunning it starts another STK application.

In [3]:
stk = STKDesktop.StartApplication(visible=True, userControl=True)
stkRoot = stk.Root
stkRoot.NewScenario('JWST_Three_Spacecraft_Approach')
scenario = stkRoot.CurrentScenario

stkRoot.UnitPreferences.ResetUnits()
stkRoot.UnitPreferences.SetCurrentUnit('DateFormat', 'UTCG')
stkRoot.UnitPreferences.SetCurrentUnit('DistanceUnit', 'km')
stkRoot.UnitPreferences.SetCurrentUnit('TimeUnit', 'sec')
stkRoot.ExecuteCommand('SetUnits / km')  # Connect commands have their own unit preferences
scenario.SetTimePeriod(start_time, stop_time)
scenario.Epoch = start_time
stkRoot.Rewind()


## 4. Create the same rotating L2 coordinate system as the previous project

Define the project’s rotating L2 frame for expressing position and velocity components.

In [4]:
# Origin: Sun/Earth-Moon L2. Axes: STK's built-in SEM_L2 axes.
# X points along the Sun-to-L2 direction; Z is normal to the associated orbital plane.
if not stkRoot.CentralBodies.Sun.Vgt.Systems.Contains('Syst_SEM_L2'):
    stkRoot.ExecuteCommand('VectorTool * CentralBody/Sun Create System Syst_SEM_L2 "Assembled" "CentralBody/Sun SEM_L2" "CentralBody/Sun SEM_L2"')
l2_system = stkRoot.CentralBodies.Sun.Vgt.Systems.Item('Syst_SEM_L2')


## 5. Load JWST and read its initial state

JWST stays in the scenario as an external-ephemeris reference. Read its initial position and velocity in the L2 frame using explicit X, Y and Z properties. All three simulated spacecraft are initialised from this state.


In [5]:
print('Loading the JWST reference ephemeris...')
if scenario.Children.Contains(AgESTKObjectType.eSatellite, 'JWST'):
    jwst = stkRoot.GetObjectFromPath('Satellite/JWST')
else:
    jwst = scenario.Children.New(AgESTKObjectType.eSatellite, 'JWST')
jwst.SetPropagatorType(AgEVePropagatorType.ePropagatorStkExternal)
jwst.Propagator.Filename = str(ephemeris_file)
jwst.Propagator.Propagate()

jwst_state = jwst.Vgt.Points.Item('Center').LocateInSystemWithRate(start_time, l2_system)
jwst_final = jwst.Vgt.Points.Item('Center').LocateInSystemWithRate(stop_time, l2_system)
if not jwst_state.IsValid or not jwst_final.IsValid:
    raise RuntimeError('Choose a scenario interval within the supplied JWST ephemeris.')
position = [jwst_state.Position.X, jwst_state.Position.Y, jwst_state.Position.Z]
velocity = [jwst_state.Velocity.X, jwst_state.Velocity.Y, jwst_state.Velocity.Z]


Loading the JWST reference ephemeris...


## 6. Create and propagate the three spacecraft

First build the original Initial_State → Coast setup over the nine-hour interval. Equal initial velocity components in the rotating L2 frame give zero initial relative velocity before the burns. Cell 9 will insert the inward burns and rerun the outer trajectories. Base receives no burn.


In [6]:
# Equal velocity components in the rotating L2 frame mean zero initial relative velocity there.
# These are INITIAL offsets only. No segment holds the spacecraft at these offsets.
spacecraft = [('Base', 0.0, 'white'), ('X_Minus', -separation_m / 2, 'cyan'), ('X_Plus', separation_m / 2, 'yellow')]

for name, x_offset_m, colour in spacecraft:
    print(f'Creating {name}: initial x offset = {x_offset_m:+.1f} m')
    if scenario.Children.Contains(AgESTKObjectType.eSatellite, name):
        satellite = stkRoot.GetObjectFromPath(f'Satellite/{name}')
    else:
        satellite = scenario.Children.New(AgESTKObjectType.eSatellite, name)
    satellite.SetPropagatorType(AgEVePropagatorType.ePropagatorAstrogator)
    mcs = satellite.Propagator.MainSequence
    mcs.RemoveAll()

    # Initial State: JWST state plus the selected X offset, converted from metres to km
    initial_state = mcs.Insert(AgEVASegmentType.eVASegmentTypeInitialState, 'Initial_State', '-')
    initial_state.SetElementType(AgEVAElementType.eVAElementTypeCartesian)
    initial_state.CoordSystemName = 'CentralBody/Sun Syst_SEM_L2'
    initial_state.OrbitEpoch = start_time
    initial_state.Element.X = position[0] + x_offset_m / 1000
    initial_state.Element.Y = position[1]
    initial_state.Element.Z = position[2]
    initial_state.Element.Vx = velocity[0]
    initial_state.Element.Vy = velocity[1]
    initial_state.Element.Vz = velocity[2]

    # Coast: propagate freely for the requested duration, with no manoeuvres or targeting
    coast = mcs.Insert(AgEVASegmentType.eVASegmentTypePropagate, 'Coast', '-')
    coast.PropagatorName = propagator_name
    coast.EnableMaxPropagationTime = False
    coast.StoppingConditions.Item('Duration').Properties.Trip = coast_seconds
    satellite.Propagator.RunMCS()

    # Align displayed body axes with L2 axes. This is attitude, not orbital station keeping.
    satellite.Attitude.Basic.SetProfileType(AgEVeProfile.eProfileFixedInAxes)
    satellite.Attitude.Basic.Profile.ReferenceAxes = 'CentralBody/Sun Syst_SEM_L2.Axes'


Creating Base: initial x offset = +0.0 m
Creating X_Minus: initial x offset = -150.0 m
Creating X_Plus: initial x offset = +150.0 m


## 7. Create the Base-centred frame and check the initial offsets

This frame follows Base's centre while retaining the L2 axis directions. Check the three initial positions and trajectory coverage. Reusing an existing frame avoids the duplicate-creation error encountered earlier.


In [7]:
# This frame follows Base's centre but keeps the same rotating L2 axis directions.
# Create the frame only if it does not already exist
if not stkRoot.CentralBodies.Sun.Vgt.Systems.Contains('Syst_Base_L2'):
    stkRoot.ExecuteCommand('VectorTool * CentralBody/Sun Create System Syst_Base_L2 "Assembled" "Satellite/Base Center" "CentralBody/Sun SEM_L2"')
base_system = stkRoot.CentralBodies.Sun.Vgt.Systems.Item('Syst_Base_L2')

# Check the propagated initial offsets and that all three trajectories reach the stop time
print('\nInitial positions relative to Base [m]:')
for name, x_offset_m, colour in spacecraft:
    satellite = stkRoot.GetObjectFromPath(f'Satellite/{name}')
    centre = satellite.Vgt.Points.Item('Center')
    state = centre.LocateInSystemWithRate(start_time, base_system)
    final_state = centre.LocateInSystemWithRate(stop_time, base_system)
    if not state.IsValid or not final_state.IsValid:
        raise RuntimeError(f'{name}: the propagated trajectory does not cover the requested interval.')
    x, y, z = state.Position.X * 1000, state.Position.Y * 1000, state.Position.Z * 1000
    if max(abs(x - x_offset_m), abs(y), abs(z)) > 0.01:
        raise RuntimeError(f'{name}: initial offset check failed: {x}, {y}, {z} m.')
    print(f'  {name:8s}: x = {x:+.3f}, y = {y:+.3f}, z = {z:+.3f}')



Initial positions relative to Base [m]:
  Base    : x = +0.000, y = +0.000, z = +0.000
  X_Minus : x = -150.000, y = -0.000, z = +0.000
  X_Plus  : x = +150.000, y = +0.000, z = +0.000


## 8. Display all four objects together

Use white for Base, cyan for X_Minus, yellow for X_Plus, and a larger **magenta star for JWST**. All four trajectories use the same Base-centred L2 frame. JWST is not unloaded or hidden. At the start, JWST and Base coincide, so their markers overlap.

The camera follows Base with the L2 reference axes. Rotate the camera approximately along Y to see the X–Z plane: X shows the inward approach and Z shows the intended miss offset. The displayed trails cover a short time window instead of showing JWST's entire multi-year ephemeris.


In [8]:
# Graphics are separate from the dynamics. A graphics warning does not invalidate propagation.
plot_objects = [(name, colour, 'Circle', 8) for name, x_offset_m, colour in spacecraft]
plot_objects += [('JWST', 'magenta', 'Star', 12)]
graphics_commands = []
for name, colour, shape, size in plot_objects:
    graphics_commands += [f'Graphics */Satellite/{name} SetColor {colour}',
        f'VO */Satellite/{name} Model Show Off',
        f'VO */Satellite/{name} Marker Show On MarkerType Line Shape {shape} Size {size}',
        f'VO */Satellite/{name} OrbitSystem Add System "CentralBody/Sun Syst_Base_L2" WindowID All',
        f'VO */Satellite/{name} OrbitSystem Modify System "InertialByWindow" WindowID All Show Off',
        f'VO */Satellite/{name} Pass3D Inherit Off OrbitLead Time {coast_seconds} OrbitTrail Time {coast_seconds}']
graphics_commands += ['VO * ViewFromTo Normal From Satellite/Base To Satellite/Base WindowID 1',
    'VO * View AddReferenceFrame "CentralBody/Sun SEM_L2 Axes" Object Satellite/Base',
    'VO * View SetReferenceFrame "CentralBody/Sun SEM_L2 Axes" WindowID 1',
    'VO * ViewFromTo Parameters Distance From 1 WindowID 1',
    'VO * ViewFromTo Reset WindowID 1']
for command in graphics_commands:
    try:
        stkRoot.ExecuteCommand(command)
    except Exception as error:
        print(f'Graphics warning: {command}\n  {error}')
stkRoot.Rewind()


## 9. Add the inward burns and propagate

Each outer spacecraft gets a single impulsive velocity change at the initial epoch. Set ΔVx = −X_offset / approach_time and ΔVz = ±miss_distance / approach_time, in L2 axes. For the defaults, the inward component is about 0.00694 m/s (6.94 mm/s). The two Z components have opposite signs.

The burn gives them velocity towards Base; it is followed by a free coast, not continued thrust or position control. Rerunning this cell updates the existing burn instead of adding another one. Fuel bookkeeping is disabled for these prescribed impulses.


In [9]:
stkRoot.UnitPreferences.SetCurrentUnit('SmallDistance', 'm')
stkRoot.UnitPreferences.SetCurrentUnit('TimeUnit', 'sec')
stkRoot.Pause()

for name, x_offset_m, colour in spacecraft:
    if name == 'Base':
        continue
    satellite = stkRoot.GetObjectFromPath(f'Satellite/{name}')
    mcs = satellite.Propagator.MainSequence

    # Insert before Coast, or reuse the existing burn when rerunning this cell
    segment_names = [mcs.Item(i).Name for i in range(mcs.Count)]
    if 'Approach_Burn' not in segment_names:
        mcs.Insert(AgEVASegmentType.eVASegmentTypeManeuver, 'Approach_Burn', 'Coast')
    burn = mcs.Item('Approach_Burn')
    burn.SetManeuverType(AgEVAManeuverType.eVAManeuverTypeImpulsive)
    impulse = burn.Maneuver
    impulse.SetAttitudeControlType(AgEVAAttitudeControl.eVAAttitudeControlThrustVector)
    impulse.UpdateMass = False

    # Aim inwards along X, with a small Z component to pass beside Base
    side = -1 if x_offset_m < 0 else 1
    dv_x = -x_offset_m / approach_seconds
    dv_z = side * miss_distance_m / approach_seconds
    thrust = impulse.AttitudeControl
    thrust.ThrustAxesName = 'CentralBody/Sun Syst_SEM_L2.Axes'
    thrust.AssignCartesian(dv_x, 0.0, dv_z)  # m/s

    satellite.Propagator.RunMCS()
    print(f'{name}: delta-V = ({dv_x:+.6f}, 0, {dv_z:+.6f}) m/s')

stkRoot.Rewind()
print(f'\nApproach expected near {approach_hours:.1f} hours. Base coasts; JWST remains an ephemeris reference.')


X_Minus: delta-V = (+0.006944, 0, -0.000231) m/s
X_Plus: delta-V = (-0.006944, 0, +0.000231) m/s

Approach expected near 6.0 hours. Base coasts; JWST remains an ephemeris reference.


## 10. Watch or replay the approach

Reset to the initial one-kilometre view and start playback with a 60-second animation step. This does not change the numerical integration step. Let the animation run before executing cell 11. Rerun this cell whenever you want to replay from the beginning.


In [13]:
stkRoot.Pause()
stkRoot.ExecuteCommand('VO * ViewFromTo Normal From Satellite/Base To Satellite/Base WindowID 1')
stkRoot.ExecuteCommand('VO * View SetReferenceFrame "CentralBody/Sun SEM_L2 Axes" WindowID 1')
stkRoot.ExecuteCommand('VO * ViewFromTo Parameters Distance From 1 WindowID 1')
stkRoot.ExecuteCommand('VO * ViewFromTo Reset WindowID 1')
stkRoot.ExecuteCommand(f'SetAnimation * StartAndCurrentTime "{start_time}" EndTime "{stop_time}" EndMode End AnimationMode Normal TimeStep 60')
stkRoot.PlayForward()


## 11. Pause near the nominal close pass and compare with JWST

Jump to six hours and print the actual distances between each outer spacecraft and Base, between the outer spacecraft, and between JWST and Base. This evaluates the nominal approach time; it does not search for the exact time of minimum separation.

Zoom to approximately 50 m from Base, increasing the viewing distance if needed to keep JWST and the two outer spacecraft within the view. JWST may have drifted from Base because the independent propagation uses the installed force model and default spacecraft properties rather than JWST's calibrated trajectory. The 5 m aiming offset is approximate, not an enforced collision-avoidance constraint.


In [14]:
stkRoot.Pause()
stkRoot.ExecuteCommand(f'SetAnimation * CurrentTime "{close_time}"')

relative_positions = {}
for name in ['X_Minus', 'X_Plus', 'JWST']:
    satellite = stkRoot.GetObjectFromPath(f'Satellite/{name}')
    state = satellite.Vgt.Points.Item('Center').LocateInSystemWithRate(close_time, base_system)
    if not state.IsValid:
        raise RuntimeError(f'{name}: state unavailable at {close_time}.')
    p = state.Position
    xyz = [p.X * 1000, p.Y * 1000, p.Z * 1000]
    relative_positions[name] = xyz
    distance_m = sum(value**2 for value in xyz)**0.5
    print(f'{name} to Base: {distance_m:.3f} m')

outer_distance_m = sum((a - b)**2 for a, b in zip(relative_positions['X_Minus'], relative_positions['X_Plus']))**0.5
print(f'Between outer spacecraft: {outer_distance_m:.3f} m')

# Include JWST in the close view, even if it is farther from Base than the outer spacecraft
max_distance_m = max(sum(value**2 for value in xyz)**0.5 for xyz in relative_positions.values())
view_distance_km = max(0.05, 3 * max_distance_m / 1000)
stkRoot.ExecuteCommand(f'VO * ViewFromTo Parameters Distance From {view_distance_km} WindowID 1')
stkRoot.ExecuteCommand('VO * ViewFromTo Reset WindowID 1')
print(f'Camera distance from Base: {view_distance_km * 1000:.1f} m')


X_Minus to Base: 5.045 m
X_Plus to Base: 5.043 m
JWST to Base: 24.202 m
Between outer spacecraft: 10.088 m
Camera distance from Base: 72.6 m


## 12. Extend JWST's trajectory from the ephemeris

Extend the scenario to 180 days from the initial epoch and reload JWST's external ephemeris.

JWST continues along the supplied trajectory; no numerical force-model propagation or additional burns are applied to it. The three simulated spacecraft retain their existing nine-hour trajectories and will therefore disappear when animation moves beyond their available data.

In [18]:
stkRoot.Pause()

# Extend the scenario for the JWST halo-orbit view
halo_days = 180
halo_seconds = halo_days * 86400
halo_stop_time = (start_date + timedelta(days=halo_days)).strftime('%d %b %Y %H:%M:%S.%f')[:-3]

scenario.StopTime = halo_stop_time
jwst = stkRoot.GetObjectFromPath('Satellite/JWST')
jwst.Propagator.Propagate()

# Check that the supplied ephemeris covers the extended interval
halo_state = jwst.Vgt.Points.Item('Center').LocateInSystemWithRate(halo_stop_time, l2_system)
if not halo_state.IsValid:
    raise RuntimeError('The JWST ephemeris does not cover this date. Reduce halo_days.')

print(f'JWST trajectory available until {halo_stop_time} UTC.')
print('The three simulated spacecraft retain their original nine-hour trajectories.')

# Extend the three simulated trajectories, retaining their existing burns
for name, x_offset_m, colour in spacecraft:
    satellite = stkRoot.GetObjectFromPath(f'Satellite/{name}')
    coast = satellite.Propagator.MainSequence.Item('Coast')
    coast.EnableMaxPropagationTime = False
    coast.StoppingConditions.Item('Duration').Properties.Trip = halo_seconds
    satellite.Propagator.RunMCS()
    print(f'{name}: propagated for {halo_days} days.')

JWST trajectory available until 04 Apr 2023 07:00:00.000 UTC.
The three simulated spacecraft retain their original nine-hour trajectories.
Base: propagated for 180 days.
X_Minus: propagated for 180 days.
X_Plus: propagated for 180 days.


## 13. Compare JWST and the three spacecraft from near Earth

Display all four trajectories in an Earth-centred frame aligned with the rotating L2 axes, then look from near Earth towards L2.

JWST follows its imported ephemeris. Base coasts freely, while X_Minus and X_Plus coast after their initial approach burns. Their trajectories can therefore depart from JWST's halo and from each other over the extended interval.

The view initially shows the positions and preceding trajectories after 180 days. Rewind and play to watch their evolution.

In [20]:
stkRoot.Pause()
stkRoot.ExecuteCommand('SetUnits / km')

# Earth-centred origin, with the rotating L2 axis directions
new_halo_frame = not stkRoot.CentralBodies.Earth.Vgt.Systems.Contains('Syst_Earth_L2')
if new_halo_frame:
    stkRoot.ExecuteCommand('VectorTool * CentralBody/Earth Create System Syst_Earth_L2 "Assembled" "CentralBody/Earth Center" "CentralBody/Sun SEM_L2"')

# Display all four trajectories in the same frame
for name in ['JWST', 'Base', 'X_Minus', 'X_Plus']:
    path = f'*/Satellite/{name}'

    try:
        stkRoot.ExecuteCommand(f'VO {path} OrbitSystem Modify System "CentralBody/Earth Syst_Earth_L2" Show On WindowID 1')
    except RuntimeError:
        stkRoot.ExecuteCommand(f'VO {path} OrbitSystem Add System "CentralBody/Earth Syst_Earth_L2" Show On WindowID 1')

    # Disable the previous trajectory frames in this window
    for system in ['CentralBody/Sun Syst_Base_L2', 'InertialByWindow', 'FixedByWindow']:
        stkRoot.ExecuteCommand(f'VO {path} OrbitSystem Modify System "{system}" Show Off WindowID 1')

    # Display the preceding trajectories and retain markers at large distances
    stkRoot.ExecuteCommand(f'VO {path} Pass3D Inherit Off OrbitLead None OrbitTrail Time {halo_seconds}')
    stkRoot.ExecuteCommand(f'VO {path} ModelDetail On ModelLabel 1e7 MarkerLabel 2e7 Marker 3e7 Point 4e7')

# Look from near Earth towards L2
stkRoot.ExecuteCommand('VO * ViewFromTo Normal From "CentralBody/Earth Center Point" To "CentralBody/Sun SEM_L2 Point" WindowID 1')
if new_halo_frame:
    stkRoot.ExecuteCommand('VO * View AddReferenceFrame "CentralBody/Sun SEM_L2 Axes" Object CentralBody')
stkRoot.ExecuteCommand('VO * View SetReferenceFrame "CentralBody/Sun SEM_L2 Axes" WindowID 1')
stkRoot.ExecuteCommand('VO * ViewFromTo Parameters Distance From 10000 UseUpVector On UpVector Z WindowID 1')
stkRoot.ExecuteCommand('Window3D * ViewVolume FieldOfView 80 MaxVisibleDist 1e7 WindowID 1')

# Show the accumulated trajectories at the end of the interval
stkRoot.ExecuteCommand(f'SetAnimation * StartTimeOnly "{start_time}" EndTime "{halo_stop_time}" CurrentTime "{halo_stop_time}" EndMode End AnimationMode Normal TimeStep 21600')
stkRoot.ExecuteCommand('VO * ViewFromTo Reset WindowID 1')

print('JWST: magenta; Base: white; X_Minus: cyan; X_Plus: yellow.')
print('All four trajectories are displayed. Rotate or widen the view if a drifting spacecraft is off-screen.')

# Uncomment to watch the trajectories develop
stkRoot.Rewind()
stkRoot.PlayForward()

JWST: magenta; Base: white; X_Minus: cyan; X_Plus: yellow.
All four trajectories are displayed. Rotate or widen the view if a drifting spacecraft is off-screen.


## 14. Save the scenario

Save the extended scenario, JWST ephemeris reference, three simulated spacecraft, approach burns, and current halo view.

Keep the saved scenario's companion files together. Save again in STK after any further viewing changes.

In [ ]:
stkRoot.Pause()

output_folder = folder / ('STK_approach_and_halo_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
output_folder.mkdir()
stkRoot.ExecuteCommand(f'Save / * "{output_folder}"')

print(f'Scenario saved in: {output_folder}')
print(f'JWST view extends to {halo_stop_time} UTC.')

## Useful STK viewing settings

- **Magenta star:** JWST. **White circle:** Base. **Cyan/yellow circles:** the two approaching spacecraft. JWST and Base start at the same position.
- The default **Sun Syst_Base_L2** Orbit System displays all four trajectories relative to Base with the L2 axis directions. The camera frame and the displayed Orbit System are separate settings.
- For a larger L2 view, add **Sun Syst_SEM_L2** under each object's **3D Graphics → Orbit System**, hide the Base-centred system in that window, and zoom out. Add the frame for **JWST as well as the three simulated spacecraft**. A nine-hour interval shows only a small section of the L2 trajectory.
- In **3D Graphics → Model**, change marker size and detail thresholds if an object becomes hard to see. Enlarged markers are symbols, not physical spacecraft dimensions.
- If the formation appears superimposed, rotate the view. Looking along Y shows the X–Z plane and the intended opposite Z miss offsets.
- Cell 11 adjusts the close-view distance to include JWST. You can manually zoom closer to the formation, but an object outside the camera field of view will then be off-screen even though its graphics remain enabled.

STK documentation: [View From/To](https://help.agi.com/stk/Content/vo/3DViewPosition.htm), [Orbit systems](https://help.agi.com/stk/Content/vo/orbitframe.htm), [Markers and models](https://help.agi.com/stk/Content/vo/model.htm), [Impulsive thrust-vector components](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVAAttitudeControlImpulsiveThrustVector.html).
